# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema, accessible at the following URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

This dataset contains mental health survey responses, demographic information, and clinical assessment scores (such as GAD-7, PHQ-9, PSQ) from residents of Kilifi County, Kenya.

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Croissant metadata and initialize the dataset object
dataset = mlc.Dataset(croissant_url)

# Print basic metadata details
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Display all available record sets in the dataset and their associated fields and `@id`s.

We'll reference entities by their `@id` as per Croissant conventions. This establishes a robust mapping between schemas and code for data extraction and processing.

In [ ]:
# List all record sets and the fields/columns by their @id
print('Record sets defined in this dataset:')
for rs in dataset.record_sets:
    print(f"@id: {rs['@id']} ; name: {rs.get('name', '')}")
    print("\tFields (by @id):")
    for field in rs['fields']:
        print(f"\t - {field['@id']} (name={field.get('name', '')})")
    if 'columns' in rs:
        print("\tColumns (by @id):")
        for col in rs['columns']:
            print(f"\t - {col['@id']} (name={col.get('name', '')})")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame.

We'll choose each record set's `@id` and reference them explicitly. See the printout above for valid record set IDs.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
print('Loading all record sets into DataFrames...')
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"Loaded {df.shape[0]} rows x {df.shape[1]} cols for record set {rs_id}")
    dataframes[rs_id] = df

# (Choose the first record set as primary for exploration)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Primary record set (@id): {main_rs_id}")
    print(f"Columns in primary record set:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print('No record sets found.')

## 4. Exploratory Data Analysis (EDA)
We demonstrate basic data processing steps: filtering, outlier removal, normalization, and grouping by demographic or clinical fields.

Reference every field using their Croissant `@id` for precision.

In [ ]:
# For demonstration, let's pick a clinically meaningful numeric field and a group field.
# Please adjust `<numeric_field_id>` and `<group_field_id>` by copying valid @id from Section 2 output!

# Example: Let's suppose GAD-7 Score is stored under @id 'gad7_score' and Gender under @id 'gender'.
numeric_field_id = 'gad7_score'  # <-- change as needed using a real @id
group_field_id = 'gender'       # <-- change as needed using a real @id

# Ensure field exists
main_df = dataframes[main_rs_id]
if numeric_field_id in main_df.columns:
    # Filter: GAD-7 score > 10
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize this numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} values (first rows):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped analysis by categorical demographic field
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std'])
        print(f"Grouped statistics for {numeric_field_id} by {group_field_id}:")
        display(grouped.head())
else:
    print(f"Field '{numeric_field_id}' not found in columns: {main_df.columns.tolist()}")

## 5. Visualization
Let's visualize the distribution of a numeric score (e.g., GAD-7) and its relationship to a demographic categorical variable.

You can adjust the field `@id` variables for alternate analyses.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id in main_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
We have loaded and explored the FAIR² Mental Health Survey Dataset using the Croissant schema and `mlcroissant`. After loading the dataset, we inspected available record sets and fields by their Croissant `@id`, loaded the primary record set, performed basic filtering and normalization of a numeric mental health score, and visualized its distribution and relationship with a demographic variable.

This workflow provides a transparent, reproducible approach for FAIR/FAIR² datasets. Update field `@id`s from the schema as needed to drill down into other demographics and survey results.